Load the raw data

First, let's load our DHS dataset and take a quick look at the first few rows to make sure everything imported correctly

In [4]:
import numpy as np
import pandas as pd

raw_path = "../data/STATcompilerExport2026930_1529.csv"
df = pd.read_csv(raw_path, skiprows=3, encoding="utf-8-sig")

print(f"Raw dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

Raw dataset loaded: 482 rows x 10 columns


,Country,Survey,By,Characteristic,Infant mortality rate,Under-five mortality rate,Fully vaccinated (8 basic antigens),Children stunted,Children wasted,Children underweight
0,Nigeria,2024 DHS,Five years preceding the survey,Total,63 (CI: 58 – 68),110 (CI: 103 – 117),NaN,NaN,NaN,NaN
1,Nigeria,2024 DHS,Ten years preceding the survey,Total,59 (CI: 55 – 63),103 (CI: 97 – 110),NaN,NaN,NaN,NaN
2,Nigeria,2024 DHS,NaN,Total,NaN,NaN,39.4,39.5,8.4,26.6
3,Nigeria,2024 DHS,Five years preceding the survey,Groups of States : North Central (L1),NaN,NaN,NaN,NaN,NaN,NaN
4,Nigeria,2024 DHS,Ten years preceding the survey,Groups of States : North Central (L1),42 (CI: 36 – 48),65 (CI: 56 – 74),NaN,NaN,NaN,NaN


We'll create a copy of the raw data to work on. This keeps our original file safe in case we need to start over.

In [5]:
df_clean = df.copy()

print("Working copy created:", df_clean.shape)

Working copy created: (482, 10)


We only want state-level data for this project, so let's keep those rows and clean up the extra text in the state names.

In [7]:
# Keep state-level records
df_clean = df_clean[
    df_clean["Characteristic"].str.startswith("States :", na=False)
].copy()

# Remove 'States :' prefix and '(L2)' suffix, then title case
df_clean["State"] = (
    df_clean["Characteristic"]
    .str.replace("States : ", "", regex=False)
    .str.replace(" (L2)", "", regex=False)
    .str.strip()
    .str.title()
)

Some indicator values were split across different rows for the same state. Let's merge them into single state rows and give the columns clear, simple names

In [8]:
# Group split indicator rows per state
df_clean = (
    df_clean.groupby(["Country", "Survey", "State"], as_index=False)
    .first()
    .drop(columns=["By", "Characteristic"], errors="ignore")
)

# Rename to clean column names
rename_map = {
    "Fully vaccinated (8 basic antigens)": "Fully_Vaccinated_Pct",
    "Children stunted": "Stunting_Pct",
    "Children wasted": "Wasting_Pct",
    "Children underweight": "Underweight_Pct",
    "Infant mortality rate": "Infant_Mortality_Rate",
    "Under-five mortality rate": "Under_Five_Mortality_Rate",
}
df_clean = df_clean.rename(columns=rename_map)

Let's check for any repeated entries and remove them so every row in our table is unique.

In [9]:
before_shape = df_clean.shape
df_clean = df_clean.drop_duplicates()

print(f"Duplicates removed: {before_shape[0] - df_clean.shape[0]}")

Duplicates removed: 0


Some percentage values are being read as plain text. Let's convert them to proper numeric numbers so Python can do calculations with them.

In [10]:
num_cols = [
    "Fully_Vaccinated_Pct",
    "Stunting_Pct",
    "Wasting_Pct",
    "Underweight_Pct",
]
for col in num_cols:
  df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

print("Data types after conversion:")
print(df_clean.dtypes)

Data types after conversion:
Country                          str
Survey                           str
State                            str
Infant_Mortality_Rate            str
Under_Five_Mortality_Rate        str
Fully_Vaccinated_Pct         float64
Stunting_Pct                 float64
Wasting_Pct                  float64
Underweight_Pct              float64
dtype: object


Let's do a quick final review to make sure there are no missing values or weird errors in our cleaned table.

In [11]:
print("===== VALIDATION REPORT =====")
print(f"Clean dataset shape: {df_clean.shape}")
print(f"Missing values:\n{df_clean.isnull().sum()}")
df_clean.head()

===== VALIDATION REPORT =====
Clean dataset shape: (111, 9)
Missing values:
Country                      0
Survey                       0
State                        0
Infant_Mortality_Rate        0
Under_Five_Mortality_Rate    0
Fully_Vaccinated_Pct         0
Stunting_Pct                 0
Wasting_Pct                  0
Underweight_Pct              0
dtype: int64


,Country,Survey,State,Infant_Mortality_Rate,Under_Five_Mortality_Rate,Fully_Vaccinated_Pct,Stunting_Pct,Wasting_Pct,Underweight_Pct
0,Nigeria,2013 DHS,Abia,79 (CI: 52 – 106),132 (CI: 98 – 166),49.8,17.3,11.0,11.6
1,Nigeria,2013 DHS,Adamawa,103 (CI: 81 – 124),174 (CI: 140 – 208),40.4,34.3,14.5,23.1
2,Nigeria,2013 DHS,Akwa Ibom,54 (CI: 35 – 73),79 (CI: 56 – 102),48.0,22.4,10.5,15.1
3,Nigeria,2013 DHS,Anambra,61 (CI: 33 – 89),87 (CI: 46 – 128),51.6,18.4,17.3,14.1
4,Nigeria,2013 DHS,Bauchi,96 (CI: 81 – 112),228 (CI: 190 – 265),6.1,50.8,23.3,40.6


Everything looks good! Let's save our final clean dataset into the data folder as a new CSV file.

In [12]:
output_path = "../data/nigeria_subnational_health_nutrition_dhs.csv"
df_clean.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Saved clean file to: {output_path}")

Saved clean file to: ../data/nigeria_subnational_health_nutrition_dhs.csv
